In [3]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly_express as px
import math
import plotly

import warnings
warnings.simplefilter('ignore')

import numpy as np
from scipy import stats

def load_numbeo_file(filepath):
    all_years = []
    xl = pd.ExcelFile(filepath)
    for sheet in xl.sheet_names:
        try:
            df = pd.read_excel(filepath, sheet_name=sheet)
            df['year'] = int(str(sheet).strip())
            all_years.append(df)
            print(f'✓ Loaded sheet: {sheet} — {len(df)} rows')
        except Exception as e:
            print(f'✗ Skipped sheet: {sheet} — {e}')
    return pd.concat(all_years, ignore_index=True)

In [9]:
# Load all 6 Numbeo files for 2025 only
cost      = pd.read_excel('../data/raw/numbeo_cost_of_living.xlsx', sheet_name='2025')
safety    = pd.read_excel('../data/raw/numbeo_crime.xlsx', sheet_name='2025')
health    = pd.read_excel('../data/raw/numbeo_health_care.xlsx', sheet_name='2025')
pollution = pd.read_excel('../data/raw/numbeo_pollution.xlsx', sheet_name='2025')
traffic   = pd.read_excel('../data/raw/numbeo_traffic.xlsx', sheet_name='2025')
property  = pd.read_excel('../data/raw/numbeo_property_prices.xlsx', sheet_name='2025')

# Merge all sub-indices on City
df = cost[['City', 'Cost of Living Index']]\
    .merge(safety[['City', 'Safety Index']], on='City')\
    .merge(health[['City', 'Health Care Index']].rename(columns={'Country': 'City'}), on='City')\
    .merge(pollution[['City', 'Pollution Index']], on='City')\
    .merge(traffic[['City', 'Traffic Index']], on='City')\
    .merge(property[['City', 'Price to Income Ratio']], on='City')

# Invert pollution and traffic (lower is better)
df['pollution_inv'] = 100 - df['Pollution Index']
df['traffic_inv']   = 100 - df['Traffic Index']
df['property_inv'] = 100 - df['Price to Income Ratio']

# Build composite quality score (average of 5 quality dimensions)
df['quality_score'] = df[['Safety Index',
                          'Health Care Index',
                          'pollution_inv',
                          'traffic_inv',
                          'property_inv']].mean(axis=1)
# Value for living index: quality divided by cost
df['value_index'] = (df['quality_score'] / df['Cost of Living Index']).round(3)

# Top 20 value for living cities
print(df[['City', 'quality_score', 'Cost of Living Index', 'value_index']]
      .sort_values('value_index', ascending=False).head(20).to_string(index=False))

                           City  quality_score  Cost of Living Index  value_index
        Nizhny Novgorod, Russia          60.28                  22.5        2.679
               Lahore, Pakistan          37.42                  18.4        2.034
               Ahmedabad, India          41.12                  21.1        1.949
Astana (Nur-Sultan), Kazakhstan          49.70                  26.9        1.848
               Curitiba, Brazil          44.44                  25.1        1.771
                 Minsk, Belarus          46.12                  26.2        1.760
                Brasov, Romania          61.02                  35.5        1.719
                  Bursa, Turkey          53.92                  31.4        1.717
             Timisoara, Romania          59.44                  34.7        1.713
               Campinas, Brazil          45.16                  27.0        1.673
          Yekaterinburg, Russia          37.48                  22.8        1.644
                

Index(['Rank', 'Country', 'Health Care Index', 'Health Care Exp. Index'], dtype='str')
